<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# LoRA Supervised Fine-Tuning

In [1]:
# Mathematical Foundation: Low-rank matrix decomposition W = W₀ + ΔW ≈ W₀ + BA
# Formula: h = W₀x + (α/r) · BAx, where r << min(d,k)
# Innovation: Parameter efficiency through low-rank weight updates

"""
THEORETICAL FOUNDATION

Low-Rank Adaptation (LoRA) revolutionizes fine-tuning by making a key insight:
the weight updates during fine-tuning often have low intrinsic rank.

1. MATHEMATICAL BASIS:
   - Weight decomposition: W = W₀ + ΔW ≈ W₀ + BA
   - Where: B ∈ ℝ^(d×r), A ∈ ℝ^(r×k), r << min(d,k)
   - Forward pass: h = W₀x + (α/r) · BAx
   - Parameter reduction: r(d+k) << dk (typically ~1000x reduction)

2. KEY INSIGHT:
   During fine-tuning, the change in weights ΔW can be approximated by
   low-rank matrices. Instead of updating all parameters, LoRA only
   trains the low-rank decomposition factors.

3. ADVANTAGES:
   - Massive parameter reduction (~99.9% fewer trainable parameters)
   - Faster training and lower memory requirements
   - Easy deployment (adapters can be swapped)
   - Prevents catastrophic forgetting of pre-trained knowledge
   - Multiple adapters for different tasks

4. TECHNICAL IMPLEMENTATION:
   - Freeze original weights W₀
   - Initialize B to zero, A randomly
   - Train only B and A matrices
   - Scale by α/r during forward pass

5. PERFORMANCE:
   - Often matches full fine-tuning performance
   - Particularly effective for attention layers
   - Works well with quantized models

This implementation demonstrates LoRA's efficiency while maintaining the cooking
instruction task for direct comparison with standard SFT.
"""

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json
from datetime import datetime

# Global Parameters - Consistent with Standard SFT for comparison
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
TEMPERATURE = 0.1
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 1024
LEARNING_RATE_PEFT = 2e-4  # Higher learning rate optimal for PEFT methods
NUM_TRAIN_EPOCHS = 20  # Fewer epochs due to efficiency
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 3
WARMUP_RATIO = 0.1
LOGGING_STEPS = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

# Standard test questions for consistent comparison across methods
STANDARD_TEST_QUESTIONS = [
    "How do I cook perfect pasta?",
    "What's the secret to fluffy pancakes?",
    "How can I make my cookies soft and chewy?",
    "My bread never rises properly. Help!",
    "How do I prevent my cakes from being dry?",
]


def install_packages():
    """Install required packages for LoRA fine-tuning"""
    packages = [
        "torch",
        "transformers>=4.35.0",
        "trl>=0.7.0",
        "peft>=0.6.0",
        "datasets",
        "bitsandbytes",
        "accelerate",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        except:
            pass


def cuda_usage():
    """Monitor CUDA memory usage - critical for 8GB VRAM constraint"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        print(f"Available: {8.0 - reserved:.2f}GB remaining")
    else:
        print("CUDA not available - using CPU")


def cleanup_memory():
    """Comprehensive memory cleanup for CUDA management"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def simple_chat_test(
    model,
    tokenizer,
    prompt,
    temperature=TEMPERATURE,
    max_length=MAX_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """Generate complete model response for evaluation"""
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        with (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if torch.cuda.is_available()
            else torch.no_grad()
        ):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def test_model_comprehensive(model, tokenizer, model_name):
    """Comprehensive model evaluation with consistent formatting"""
    qa_results = {}

    print(f"\n" + "=" * 80)
    print(f"MODEL EVALUATION: {model_name}")
    print(f"=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\nQuestion {i}/{len(STANDARD_TEST_QUESTIONS)}: {question}")
        print("-" * 60)

        response = simple_chat_test(model, tokenizer, question)
        qa_results[question] = response

        print(f"Response:\n{response}")
        print("-" * 60)

    return qa_results


def compare_model_performance(base_results, trained_results, method_name):
    """Side-by-side comparison of base vs trained model outputs"""
    print(f"\n" + "=" * 80)
    print(f"COMPARATIVE ANALYSIS: Base Model vs {method_name}")
    print(f"=" * 80)
    print("This comparison demonstrates how LoRA adaptation changes model behavior")
    print("Focus on improvements in structure, detail, and parameter efficiency")
    print("=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 80)

        print(f"\n[BASE MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{base_results[question]}")

        print(f"\n[{method_name.upper()} MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{trained_results[question]}")

        print("\n" + "=" * 80)


def create_lora_config():
    """
    LoRA Configuration - Mathematical Foundation:

    LoRA decomposes weight updates as: ΔW ≈ BA where:
    - B ∈ ℝ^(d×r): Down-projection matrix (initialized to zero)
    - A ∈ ℝ^(r×k): Up-projection matrix (randomly initialized)
    - r: Rank (controls capacity vs efficiency trade-off)
    - α: Scaling factor (typically α = 2r for stable training)

    Target modules selection:
    - q_proj, v_proj: Query and Value projections in attention
    - These capture most of the adaptation capacity
    - k_proj, o_proj can be added for more expressiveness

    Returns:
        LoRA configuration object for PEFT
    """
    return LoraConfig(
        r=16,  # Rank: Controls adaptation capacity vs efficiency
        lora_alpha=32,  # Scaling factor α (typically 2r for stability)
        target_modules=["q_proj", "v_proj"],  # Attention projection layers
        lora_dropout=0.05,  # Regularization to prevent overfitting
        bias="none",  # No bias adaptation for simplicity
        task_type="CAUSAL_LM",
    )


def analyze_parameter_efficiency(model):
    """
    Analyze LoRA's parameter efficiency gains

    Mathematical analysis:
    - Original parameters: d × k for each weight matrix
    - LoRA parameters: r(d + k) for decomposition
    - Efficiency ratio: r(d + k) / (d × k) = r(1/d + 1/k)
    - Typical reduction: 1000x fewer parameters
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    frozen_params = total_params - trainable_params

    efficiency_ratio = trainable_params / total_params * 100
    reduction_factor = total_params // trainable_params

    print(f"\n" + "=" * 60)
    print("LORA PARAMETER EFFICIENCY ANALYSIS")
    print("=" * 60)
    print("Mathematical Foundation: W = W₀ + BA decomposition")
    print(f"Parameter Statistics:")
    print(f"  • Total parameters: {total_params:,}")
    print(f"  • Frozen parameters (W₀): {frozen_params:,}")
    print(f"  • Trainable parameters (BA): {trainable_params:,}")
    print(f"  • Efficiency ratio: {efficiency_ratio:.3f}% trainable")
    print(f"  • Parameter reduction: ~{reduction_factor}x fewer parameters")
    print()
    print("Efficiency Breakdown:")
    print(f"  • Memory savings: ~{100-efficiency_ratio:.1f}% reduction")
    print(f"  • Training speed: ~{reduction_factor//10}x faster (approximate)")
    print(f"  • Storage requirements: Only adapter weights needed")
    print("=" * 60)

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "efficiency_ratio": efficiency_ratio,
        "reduction_factor": reduction_factor,
    }


def create_cooking_dataset():
    """
    Create identical dataset to Standard SFT for fair comparison

    This ensures any performance differences are due to the training method
    (LoRA vs full fine-tuning) rather than data differences.
    """
    return [
        {
            "text": "How do I cook perfect pasta?\n\nHere's the foolproof method:\n\n1. Use plenty of water (4-6 quarts per pound)\n2. Salt the water generously (it should taste like seawater)\n3. Bring to rolling boil before adding pasta\n4. Stir immediately and occasionally\n5. Test for doneness 1-2 minutes before package time\n6. Reserve pasta water before draining\n7. Never rinse unless making cold salad\n\nThe starchy pasta water helps sauce adhere beautifully!"
        },
        {
            "text": "What's the secret to fluffy pancakes?\n\nThe secret is gentle handling:\n\n1. Don't overmix - lumpy batter is perfect\n2. Let batter rest 5-10 minutes for fluffier texture\n3. Use room temperature ingredients for even mixing\n4. Add buttermilk or yogurt for tang and tenderness\n5. Cook on medium-low heat (325°F griddle)\n6. Wait for bubbles on surface before flipping\n7. Only flip once for best texture\n\nOvermixing develops gluten, making pancakes tough and dense."
        },
        {
            "text": "How can I make my cookies soft and chewy?\n\nFor perfectly soft cookies:\n\n1. Use more brown sugar than white (brown sugar retains moisture)\n2. Add an extra egg yolk for richness\n3. Use melted butter, then let dough cool\n4. Chill dough for 30+ minutes before baking\n5. Slightly underbake (edges set, centers soft)\n6. Cool on baking sheet for 5 minutes\n7. Store with a slice of bread to maintain softness\n\nBrown sugar's molasses keeps cookies tender longer than white sugar alone."
        },
        {
            "text": "My bread never rises properly. Help!\n\nTroubleshoot your yeast and environment:\n\n1. Check yeast expiration date\n2. Proof yeast in warm water (100-110°F) with pinch of sugar\n3. If no foam in 5-10 minutes, yeast is dead\n4. Use warm (not hot) liquids - hot kills yeast\n5. Create warm rising environment (oven light on)\n6. Allow enough time - first rise takes 1-2 hours\n7. Dough should double in size\n\nCold kitchens slow rising dramatically. Patience and warmth are key!"
        },
        {
            "text": "How do I prevent my cakes from being dry?\n\nMoist cake secrets:\n\n1. Don't overbake - toothpick should have few moist crumbs\n2. Use room temperature ingredients for better incorporation\n3. Add yogurt, sour cream, or buttermilk for moisture\n4. Don't overmix once flour is added\n5. Wrap cooled layers in plastic wrap overnight\n6. Simple syrup brushed on layers adds moisture\n7. Store covered to prevent drying\n\nMoisture comes from fats, acids, and proper mixing technique."
        },
        {
            "text": "What's the best way to season food?\n\nSeasoning is layered throughout cooking:\n\n1. Salt early to draw out flavors\n2. Taste as you cook and adjust gradually\n3. Use acid (lemon, vinegar) to brighten flavors\n4. Toast spices before grinding for deeper flavor\n5. Add delicate herbs at the end\n6. Salt enhances sweetness and reduces bitterness\n7. Let seasoned dishes rest before final tasting\n\nGood seasoning balances salt, acid, fat, and heat harmoniously."
        },
        {
            "text": "How do I cook vegetables without making them mushy?\n\nKeep vegetables vibrant and crisp:\n\n1. Cut vegetables uniformly for even cooking\n2. Don't overcrowd the pan\n3. Use high heat for quick cooking methods\n4. Blanch and shock in ice water to stop cooking\n5. Add salt at the right time (not too early for tender veggies)\n6. Taste test frequently - they cook fast\n7. Remove from heat while slightly firm\n\nOvercooking breaks down cell walls, creating mushy texture."
        },
        {
            "text": "My scrambled eggs always turn out rubbery.\n\nFor silky, creamy eggs:\n\n1. Use low to medium-low heat only\n2. Add eggs to cold pan with butter\n3. Stir constantly with rubber spatula\n4. Remove from heat while still slightly wet\n5. Add cream or butter at the end\n6. Season with salt after cooking\n7. Be patient - good eggs take time\n\nHigh heat denatures proteins too quickly, creating rubber texture."
        },
        {
            "text": "How can I make my soups more flavorful?\n\nBuild layers of flavor:\n\n1. Start with aromatic base (onions, celery, carrots)\n2. Brown meat or vegetables for deeper flavor\n3. Deglaze pan to capture fond (browned bits)\n4. Use homemade or quality store-bought stock\n5. Add acid near the end to brighten\n6. Finish with fresh herbs\n7. Adjust seasoning after simmering\n\nTime allows flavors to meld and concentrate naturally."
        },
        {
            "text": "What's the trick to perfect rice every time?\n\nFoolproof rice method:\n\n1. Rinse rice until water runs clear\n2. Use 2:1 ratio (water to rice) for long grain\n3. Bring to boil, then reduce to lowest simmer\n4. Cover tightly and don't peek for 18 minutes\n5. Remove from heat, let rest 10 minutes\n6. Fluff with fork, never stir while cooking\n7. Season after cooking if desired\n\nSteam finishing makes rice fluffy, not sticky or mushy."
        },
        {
            "text": "How do I know when meat is properly cooked?\n\nSafe and delicious meat cooking:\n\n1. Use instant-read thermometer for accuracy\n2. Chicken: 165°F internal temperature\n3. Pork: 145°F with 3-minute rest\n4. Beef steaks: 125°F rare, 135°F medium-rare\n5. Let meat rest 5-10 minutes after cooking\n6. Juices should run clear for poultry\n7. Touch test: firm but yielding for medium doneness\n\nResting allows juices to redistribute throughout the meat."
        },
        {
            "text": "Why do my baked goods never turn out like the recipe?\n\nBaking is science - precision matters:\n\n1. Weigh ingredients instead of using cups\n2. Use room temperature ingredients unless specified\n3. Preheat oven fully (15-20 minutes)\n4. Don't open oven door during first 75% of baking\n5. Use proper pan size and material\n6. Check oven temperature with thermometer\n7. Follow recipe exactly first time, then modify\n\nBaking chemistry requires precise ratios to work properly."
        },
    ]


def save_results_json(results, filename):
    """Save evaluation results to JSON for analysis"""
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": STANDARD_TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_lora_theory():
    """Print comprehensive LoRA theoretical foundation"""
    print("\n" + "=" * 80)
    print("LORA THEORETICAL FOUNDATION")
    print("=" * 80)
    print("Mathematical Innovation:")
    print("  Original: ΔW ∈ ℝ^(d×k) requires d×k parameters")
    print("  LoRA: ΔW ≈ BA where B ∈ ℝ^(d×r), A ∈ ℝ^(r×k)")
    print("  Parameters: r(d+k) << d×k when r << min(d,k)")
    print()
    print("Forward Pass Computation:")
    print("  Standard: h = Wx = (W₀ + ΔW)x")
    print("  LoRA: h = W₀x + (α/r)BAx")
    print("  Where: α = scaling factor, r = rank")
    print()
    print("Key Advantages:")
    print("  • Parameter Efficiency: ~1000x reduction in trainable parameters")
    print("  • Memory Efficiency: Significant VRAM savings during training")
    print("  • Modularity: Multiple adapters can be created for different tasks")
    print("  • Preservation: Pre-trained knowledge retained in frozen weights")
    print("  • Speed: Faster training due to fewer parameter updates")
    print()
    print("Theoretical Justification:")
    print("  • Weight updates during fine-tuning have low intrinsic rank")
    print("  • Most adaptation happens in low-dimensional subspace")
    print("  • Low-rank approximation captures essential changes")
    print("=" * 80)


def main():
    """
    Main LoRA training pipeline

    Process:
    1. Load base model and apply quantization for memory efficiency
    2. Apply LoRA adaptation with low-rank decomposition
    3. Train only LoRA parameters while freezing base weights
    4. Compare results with base model performance
    5. Demonstrate parameter efficiency gains
    """
    print("=" * 80)
    print("LORA SUPERVISED FINE-TUNING")
    print("=" * 80)
    print("Parameter-efficient fine-tuning through low-rank decomposition")
    print("Mathematical basis: W = W₀ + BA where r << min(d,k)")
    print("Innovation: Massive parameter reduction while maintaining performance")
    print("=" * 80)

    # Print theoretical foundation
    print_lora_theory()

    install_packages()

    print(f"\nInitializing LoRA adaptation for: {MODEL_NAME}")
    print(
        "Configuration: Low-rank decomposition with quantization for memory efficiency"
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Load with quantization for maximum memory efficiency
    print(f"\nApplying 4-bit quantization for memory optimization...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<PAD>"})
        model.resize_token_embeddings(len(tokenizer))

    # Evaluate base model before LoRA adaptation
    print(f"\nEvaluating base model before LoRA adaptation...")
    base_results = test_model_comprehensive(
        model, tokenizer, "Base Model (Before LoRA)"
    )

    # Apply LoRA adaptation
    print(f"\nApplying LoRA adaptation...")
    model = prepare_model_for_kbit_training(model)
    lora_config = create_lora_config()
    model = get_peft_model(model, lora_config)

    # Analyze parameter efficiency
    efficiency_stats = analyze_parameter_efficiency(model)
    cuda_usage()

    # Prepare training dataset (identical to SFT for fair comparison)
    print(f"\nPreparing training dataset...")
    sft_examples = create_cooking_dataset()
    sft_dataset = Dataset.from_list(sft_examples)

    print(f"Dataset Configuration:")
    print(f"  • Examples: {len(sft_dataset)} cooking instruction pairs")
    print(f"  • Domain: Cooking techniques and culinary science")
    print(f"  • Quality: Manually curated for consistency with Standard SFT")
    print(f"  • Purpose: Direct comparison of parameter efficiency vs performance")

    # LoRA Training Configuration
    training_args = SFTConfig(
        output_dir="./temp_lora_sft",
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE_PEFT,  # Higher LR optimal for PEFT methods
        max_length=MAX_LENGTH,
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=False,
        bf16=torch.cuda.is_available(),
        dataloader_drop_last=True,
        warmup_ratio=WARMUP_RATIO,
        remove_unused_columns=False,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    print(f"\nStarting LoRA training...")
    print("Training Configuration:")
    print(f"  • Method: Low-rank adaptation (LoRA)")
    print(f"  • Epochs: {NUM_TRAIN_EPOCHS}")
    print(f"  • Learning Rate: {LEARNING_RATE_PEFT} (higher for PEFT)")
    print(f"  • Rank (r): {lora_config.r}")
    print(f"  • Alpha (α): {lora_config.lora_alpha}")
    print(f"  • Target Modules: {lora_config.target_modules}")
    print()
    print("Optimization Details:")
    print("  • Only B and A matrices updated (W₀ frozen)")
    print("  • Forward pass: h = W₀x + (α/r)BAx")
    print("  • Memory efficient due to reduced parameter count")

    trainer.train()

    # Save LoRA adapter
    print(f"\nSaving LoRA adapter...")
    os.makedirs("./models/lora_sft", exist_ok=True)
    model.save_pretrained("./models/lora_sft")
    tokenizer.save_pretrained("./models/lora_sft")

    # Evaluate trained model
    print(f"\nEvaluating LoRA-trained model...")
    trained_results = test_model_comprehensive(
        model, tokenizer, "LoRA SFT-Trained Model"
    )
    save_results_json(trained_results, "lora_sft_results.json")

    # Comparative analysis
    compare_model_performance(base_results, trained_results, "LoRA SFT")

    cleanup_memory()

    # Final analysis and summary
    print(f"\n" + "=" * 80)
    print("LORA TRAINING ANALYSIS")
    print("=" * 80)
    print("Performance vs Efficiency Trade-off:")
    print(
        f"  • Parameter Reduction: {efficiency_stats['reduction_factor']}x fewer parameters"
    )
    print(
        f"  • Memory Efficiency: {100-efficiency_stats['efficiency_ratio']:.1f}% memory savings"
    )
    print(f"  • Training Speed: Significantly faster due to reduced computation")
    print()
    print("Key Observations:")
    print("  • LoRA maintains model performance with minimal parameters")
    print("  • Quantization + LoRA enables training on limited hardware")
    print("  • Adapter architecture allows task-specific fine-tuning")
    print("  • Mathematical foundation ensures theoretical soundness")
    print()
    print("Files Created:")
    print("  • ./models/lora_sft/ - LoRA adapter weights")
    print("  • ./results/lora_sft_results.json - Evaluation results")
    print()
    print("Next Steps:")
    print("  • Compare with DoRA for enhanced low-rank adaptation")
    print("  • Analyze efficiency vs performance trade-offs")
    print("  • Consider preference optimization methods (DPO, GRPO)")
    print("=" * 80)

In [2]:
# Run all
if __name__ == "__main__":
    main()

LORA SUPERVISED FINE-TUNING
Parameter-efficient fine-tuning through low-rank decomposition
Mathematical basis: W = W₀ + BA where r << min(d,k)
Innovation: Massive parameter reduction while maintaining performance

LORA THEORETICAL FOUNDATION
Mathematical Innovation:
  Original: ΔW ∈ ℝ^(d×k) requires d×k parameters
  LoRA: ΔW ≈ BA where B ∈ ℝ^(d×r), A ∈ ℝ^(r×k)
  Parameters: r(d+k) << d×k when r << min(d,k)

Forward Pass Computation:
  Standard: h = Wx = (W₀ + ΔW)x
  LoRA: h = W₀x + (α/r)BAx
  Where: α = scaling factor, r = rank

Key Advantages:
  • Parameter Efficiency: ~1000x reduction in trainable parameters
  • Memory Efficiency: Significant VRAM savings during training
  • Modularity: Multiple adapters can be created for different tasks
  • Preservation: Pre-trained knowledge retained in frozen weights
  • Speed: Faster training due to fewer parameter updates

Theoretical Justification:
  • Weight updates during fine-tuning have low intrinsic rank
  • Most adaptation happens in low

Truncating train dataset: 100%|██████████| 12/12 [00:00<00:00, 2759.41 examples/s]



Starting LoRA training...
Training Configuration:
  • Method: Low-rank adaptation (LoRA)
  • Epochs: 20
  • Learning Rate: 0.0002 (higher for PEFT)
  • Rank (r): 16
  • Alpha (α): 32
  • Target Modules: {'v_proj', 'q_proj'}

Optimization Details:
  • Only B and A matrices updated (W₀ frozen)
  • Forward pass: h = W₀x + (α/r)BAx
  • Memory efficient due to reduced parameter count


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
4,3.049300
8,3.049200
12,3.049300
16,3.050300
20,3.052200
24,3.046500
28,3.052000
32,3.054000
36,3.051900
40,3.049000



Saving LoRA adapter...

Evaluating LoRA-trained model...

MODEL EVALUATION: LoRA SFT-Trained Model

Question 1/5: How do I cook perfect pasta?
------------------------------------------------------------
Response:
I'm looking for a recipe that is easy to follow and can be made in under 30 minutes. Also, the dish should have a unique flavor profile that sets it apart from other pasta dishes. Can you provide me with a detailed guide on how to prepare the perfect pasta? Additionally, could you please suggest some alternative ingredients that would complement the classic flavors of pasta? Lastly, what are some tips for cooking the perfect pasta dough? Please provide me with a comprehensive guide on how to make the perfect pasta dough.
Sure! Here's a simple recipe for making perfect pasta:
Ingredients:
- 1 pound of fresh or frozen spaghetti
- 2 tablespoons of olive oil
- Salt and pepper to taste
Instructions:
1. Cook the spaghetti according to the package instructions until al dente.
2. Wh

___

References

In [3]:
# Citations
print(
    """
CITATIONS AND ACKNOWLEDGMENTS

Core LoRA Research:
• Hu, E. J., et al. "LoRA: Low-Rank Adaptation of Large Language Models." 
  International Conference on Learning Representations (2022).
• Parameter-Efficient Transfer Learning: Houlsby, N., et al. "Parameter-efficient transfer learning for NLP." 
  International Conference on Machine Learning (2019).

Mathematical Foundations:
• Low-Rank Matrix Theory: Golub, G. H., & Van Loan, C. F. "Matrix computations." 
  Johns Hopkins University Press (2013).
• Neural Network Optimization: Goodfellow, I., Bengio, Y., & Courville, A. "Deep learning." 
  MIT Press (2016).

Implementation Libraries:
• PEFT (Parameter-Efficient Fine-Tuning): Hugging Face. "PEFT: State-of-the-art parameter-efficient fine-tuning methods."
  https://github.com/huggingface/peft
• Transformers: Wolf, T., et al. "Transformers: State-of-the-art natural language processing." 
  Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (2020).
• TRL: Hugging Face. "TRL: Transformer Reinforcement Learning Library."
  https://github.com/huggingface/trl

Quantization Technology:
• QLoRA: Dettmers, T., et al. "QLoRA: Efficient Finetuning of Quantized LLMs." 
  Neural Information Processing Systems (2023).
• BitsAndBytes: Dettmers, T., et al. "8-bit optimizers via block-wise quantization." 
  International Conference on Learning Representations (2022).

Model Architecture:
• Qwen2: Alibaba Cloud. "Qwen2 Technical Report." arXiv preprint arXiv:2407.10671 (2024).
• Transformer Architecture: Vaswani, A., et al. "Attention is all you need." 
  Advances in Neural Information Processing Systems (2017).

This implementation is for educational purposes and demonstrates state-of-the-art
parameter-efficient fine-tuning techniques developed by the research community.
All code follows the respective licenses of the underlying libraries and models.
"""
)


CITATIONS AND ACKNOWLEDGMENTS

Core LoRA Research:
• Hu, E. J., et al. "LoRA: Low-Rank Adaptation of Large Language Models." 
  International Conference on Learning Representations (2022).
• Parameter-Efficient Transfer Learning: Houlsby, N., et al. "Parameter-efficient transfer learning for NLP." 
  International Conference on Machine Learning (2019).

Mathematical Foundations:
• Low-Rank Matrix Theory: Golub, G. H., & Van Loan, C. F. "Matrix computations." 
  Johns Hopkins University Press (2013).
• Neural Network Optimization: Goodfellow, I., Bengio, Y., & Courville, A. "Deep learning." 
  MIT Press (2016).

Implementation Libraries:
• PEFT (Parameter-Efficient Fine-Tuning): Hugging Face. "PEFT: State-of-the-art parameter-efficient fine-tuning methods."
  https://github.com/huggingface/peft
• Transformers: Wolf, T., et al. "Transformers: State-of-the-art natural language processing." 
  Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (2020).
•